<a href="https://colab.research.google.com/github/gmauricio-toledo/tda-gdl/blob/main/04-Efecto_del_preprocesamiento_clasificacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Efectos del preprocesamiento en una tarea de clasificación

Registraremos la duración

In [ ]:
from time import time

In [ ]:
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(x_train.shape[0],-1)
x_test = x_test.reshape(x_test.shape[0],-1)

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

## Estrategia 1: Modelo directo

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)

start_time = time()
knn.fit(x_train, y_train)
end_time = time()
print(f"Tiempo de entrenamiento: {end_time - start_time} segundos")

start_time = time()
y_train_pred = knn.predict(x_train)
y_test_pred = knn.predict(x_test)
end_time = time()
print(f"Tiempo de predicción: {end_time - start_time} segundos")

start = time()
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
end = time()
print(f"Tiempo de evaluación: {end - start} segundos")

print(f"Train accuracy: {train_accuracy}")
print(f"Test accuracy: {test_accuracy}")

## Estrategia 2: Preprocesamiento

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

scaler = StandardScaler()

start = time()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
end = time()
print(f"Tiempo de preprocesamiento: {end - start} segundos")

knn = KNeighborsClassifier(n_neighbors=5)

start = time()
knn.fit(x_train_scaled, y_train)
end = time()
print(f"Tiempo de entrenamiento: {end - start} segundos")

start = time()
y_train_pred = knn.predict(x_train_scaled)
y_test_pred = knn.predict(x_test_scaled)
end = time()
print(f"Tiempo de predicción: {end - start} segundos")

start = time()
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
end = time()
print(f"Tiempo de evaluación: {end - start} segundos")

print(f"Train accuracy: {train_accuracy}")
print(f"Test accuracy: {test_accuracy}")

🔵 ¿Por qué bajó?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

idxs = np.random.choice(x_train.shape[0], size=3, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 4))

for i, idx in enumerate(idxs):
    axes[0,i].imshow(x_train[idx].reshape(28, 28), cmap='gray')
    axes[1,i].imshow(x_train_scaled[idx].reshape(28, 28), cmap='gray')
    axes[0,i].axis('off')
    axes[1,i].axis('off')
fig.show()

## Estrategia 3: Proyección en un subespacio

In [ ]:
import numpy as np

rangos_variables_train = np.max(x_train,axis=0) - np.min(x_train,axis=0)
zero_idxs_train = np.where(rangos_variables_train == 0)[0]
non_zero_idxs_train = np.where(rangos_variables_train != 0)[0]
print(f"columnas identicamente cero: {zero_idxs_train}")



In [ ]:
x_train_proj = x_train[:,non_zero_idxs_train]
x_train_proj.shape

🔵 ¿Por qué eliminamos las columnas calculadas con el conjunto de entrenamiento?

In [ ]:
x_test_proj = x_test[:,non_zero_idxs_train]
x_test_proj.shape

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from time import time
from sklearn.metrics import accuracy_score

clf = KNeighborsClassifier(n_neighbors=5)

start = time()
clf.fit(x_train_proj, y_train)
end = time()

print(f"Tiempo de entrenamiento: {end - start} segundos")

start = time()
y_train_pred = clf.predict(x_train_proj)
y_test_pred = clf.predict(x_test_proj)
end = time()

print(f"Tiempo de predicción: {end - start} segundos")

start = time()
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
end = time()
print(f"Tiempo de evaluación: {end - start} segundos")

print(f"Train accuracy: {train_accuracy}")
print(f"Test accuracy: {test_accuracy}")

## Estrategia 4: Umbral de varianza

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.15)
x_train_var = selector.fit_transform(x_train)
x_test_var = selector.transform(x_test)

In [ ]:
x_train_var.shape

In [ ]:
selector.get_support()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from time import time
from sklearn.metrics import accuracy_score

clf = KNeighborsClassifier(n_neighbors=5)

start = time()
clf.fit(x_train_var, y_train)
end = time()

print(f"Tiempo de entrenamiento: {end - start} segundos")

start = time()
y_train_pred = clf.predict(x_train_var)
y_test_pred = clf.predict(x_test_var)
end = time()

print(f"Tiempo de predicción: {end - start} segundos")

start = time()
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
end = time()
print(f"Tiempo de evaluación: {end - start} segundos")

print(f"Train accuracy: {train_accuracy}")
print(f"Test accuracy: {test_accuracy}")

## Estrategia 5: PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from time import time
from sklearn.metrics import accuracy_score

In [ ]:
pca = PCA(n_components=50)
x_train_pca = pca.fit_transform(x_train)
x_test_pca = pca.transform(x_test)

In [ ]:
clf = KNeighborsClassifier(n_neighbors=5)

start = time()
clf.fit(x_train_pca, y_train)
end = time()

print(f"Tiempo de entrenamiento: {end - start} segundos")

start = time()
y_train_pred = clf.predict(x_train_pca)
y_test_pred = clf.predict(x_test_pca)
end = time()

print(f"Tiempo de predicción: {end - start} segundos")

start = time()
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
end = time()
print(f"Tiempo de evaluación: {end - start} segundos")

print(f"Train accuracy: {train_accuracy}")
print(f"Test accuracy: {test_accuracy}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
x_train_pca = pca.fit_transform(x_train)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Components')
plt.ylabel('Varianza acumulada')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure()
for i in range(10):
    plt.scatter(x_train_pca[y_train==i,0],x_train_pca[y_train==i,1],label=i)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()